# IELTS Question Generation

**Inputs (source controls):** `PassageContext`, `QuestionType`, `category`, `difficulty`  
**Outputs (targets):** `InstructionContext`, `Question`, `Answer`

What you get:
- Robust loader for **CSV / JSON / JSONL** (`load_table`)
- Schema normalization & filtering (never learns placeholders)
- Special tokens: `<DIFF_EASY|MEDIUM|HARD>`, `<CAT_*>`, `<TYPE_*>`, `<SEP>`, `<BLANK>`
- Grouped split by `passage_id` (leakage-safe)
- Early stopping, label smoothing, warmup; ROUGE-L sanity metric
- Works on older Transformers: `predict_with_generate` set on **Trainer** (not `TrainingArguments`)
- Optional **LoRA (PEFT)** for low VRAM
- Demo generation at the end

> Set `CFG["data_path"]` to your dataset path (e.g., `/kaggle/working/final_data.json`), then run sequentially.

In [1]:
!pip install transformers datasets sentencepiece accelerate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 40.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 4.9 MB/s eta 0:00:000:00:0100:01
ERROR: pip's de

In [2]:
!pip install rouge-score

In [3]:
!gdown --id 1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx
To: /kaggle/working/final_data.json
100%|██████████████████████████████████████| 15.4M/15.4M [00:00<00:00, 72.0MB/s]


In [5]:
import os, re, hashlib, random, json
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset
from sklearn.model_selection import GroupShuffleSplit

from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

try:
    from rouge_score import rouge_scorer
except Exception:
    rouge_scorer = None

# PEFT (optional)
HAVE_PEFT = False
try:
    from peft import LoraConfig, get_peft_model, TaskType
    HAVE_PEFT = True
except Exception:
    pass

# Item Response Theory (IRT) Proxy (Scenario 2)
IRT_PROXY_MAP = {
    "Easy": -1.0,
    "Medium": 0.0,
    "Hard": 1.0
}
DEFAULT_B_VALUE = 0.0
DEFAULT_A_VALUE = 1.0
DEFAULT_C_VALUE = 0.0

# Control tokens
UTILITY_TOKENS = ["<SEP>", "<BLANK>", "<END>"]

print("✔️ Imports loaded. PEFT available:", HAVE_PEFT)
print(f"✔️ IRT Proxy Map (Scenario 2) Enabled: {IRT_PROXY_MAP}")

# ==== CONFIG ====
CFG = {
    # Data
    "data_path": "/kaggle/working/final_data.json",
    "output_dir": "./controlable-question-generation-GPT2",
    
    # Model
    "base_model": "gpt2",
    
    # Training
    "epochs": 30,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "lr": 5e-5,
    "warmup_ratio": 0.15,
    "weight_decay": 0.01,
    "patience": 7,
    
    # Sequence lengths
    "max_source_len": 512,
    "max_target_len": 256,
    "max_total_len": 768,
    
    # Generation
    "num_beams": 4,
    "length_penalty": 1.0,
    "temperature": 0.7,
    "top_p": 0.9,
    
    # Misc
    "seed": 42,
    "fp16": True,
    "bf16": False,
    "grad_ckpt": False,
    
    # LoRA
    "use_lora": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    
    # Curriculum Learning
    "use_curriculum": True,
    "curriculum_stages": [
        {"name": "stage1_easy", "epochs": 5, "filter": ["Easy"], "types": ["readingShortAnswer"]},
        {"name": "stage2_medium", "epochs": 5, "filter": ["Easy", "Medium"], "types": ["readingShortAnswer", "readingTrueFalseNotGiven"]},
        {"name": "stage3_all", "epochs": 15, "filter": None, "types": None},
    ],
    
    # Data Augmentation
    "use_augmentation": False,
    "augmentation_factor": 2.0,
}

print("✔️ Config:", CFG)

✔️ Imports loaded. PEFT available: True
✔️ IRT Proxy Map (Scenario 2) Enabled: {'Easy': -1.0, 'Medium': 0.0, 'Hard': 1.0}
✔️ Config: {'data_path': '/kaggle/working/final_data.json', 'output_dir': './controlable-question-generation-GPT2', 'base_model': 'gpt2', 'epochs': 30, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'lr': 5e-05, 'warmup_ratio': 0.15, 'weight_decay': 0.01, 'patience': 7, 'max_source_len': 512, 'max_target_len': 256, 'max_total_len': 768, 'num_beams': 4, 'length_penalty': 1.0, 'temperature': 0.7, 'top_p': 0.9, 'seed': 42, 'fp16': True, 'bf16': False, 'grad_ckpt': False, 'use_lora': True, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'use_curriculum': True, 'curriculum_stages': [{'name': 'stage1_easy', 'epochs': 5, 'filter': ['Easy'], 'types': ['readingShortAnswer']}, {'name': 'stage2_medium', 'epochs': 5, 'filter': ['Easy', 'Medium'], 'types': ['readingShortAnswer', 'readingTrueFalseNotGiven']}, {'name': 'stage3_all', 'epochs': 15, 'filter': None, 'types':

In [6]:
# ==== HELPER FUNCTIONS ====
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def stable_passage_id(row: pd.Series) -> str:
    if "passage_id" in row and pd.notna(row["passage_id"]):
        return str(row["passage_id"])
    txt = str(row["passage"]).strip()
    return hashlib.md5(txt.encode("utf-8")).hexdigest()

def _canon(s: str) -> str:
    s = str(s).strip().upper().replace("-", "_").replace(" ", "_")
    return "".join(ch for ch in s if ch.isalnum() or ch == "_")

def normalize_sep_blank(text: str) -> str:
    if not isinstance(text, str):
        return text
    out = re.sub(r"\\bSEP\\b", "<SEP>", text)
    out = re.sub(r"\\bBLANK\\b", "<BLANK>", out)
    return out

def load_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in (".json", ".jsonl"):
        try:
            df = pd.read_json(path, lines=(ext == ".jsonl"))
        except ValueError:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, dict) and "data" in obj:
                df = pd.DataFrame(obj["data"])
            elif isinstance(obj, list):
                df = pd.DataFrame(obj)
            else:
                raise ValueError("Unsupported JSON structure")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return df

def grouped_splits(df: pd.DataFrame, holdout_frac: float = 0.20, seed: int = 42):
    groups = df["passage_id"].values
    gss = GroupShuffleSplit(n_splits=1, test_size=holdout_frac, random_state=seed)
    train_idx, holdout_idx = next(gss.split(df, groups=groups))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_hold = df.iloc[holdout_idx].reset_index(drop=True)
    
    groups_hold = df_hold["passage_id"].values
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_idx, test_idx = next(gss2.split(df_hold, groups=groups_hold))
    df_val = df_hold.iloc[val_idx].reset_index(drop=True)
    df_test = df_hold.iloc[test_idx].reset_index(drop=True)
    
    return df_train, df_val, df_test

# Control tokens
def make_control_tokens_from_df(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    qts = []
    if "question_type" in df.columns:
        qts = sorted({_canon(t) for t in df["question_type"].dropna().tolist()})
    elif "qtype" in df.columns:
        qts = sorted({_canon(t) for t in df["qtype"].dropna().tolist()})
    
    cat_tokens = []
    type_tokens = [f"<TYPE_{t}>" for t in qts if t]
    return cat_tokens, type_tokens

def as_type_token(qtype: Optional[str]) -> str:
    if not isinstance(qtype, str) or not qtype.strip():
        return ""
    return f"<TYPE_{_canon(qtype)}>"

print("✅ Helper functions defined")

✅ Helper functions defined


In [7]:
# ==== PREFIX MAPPING ====
PREFIX_MAP = {
    'READINGTRUEFALSENOTGIVEN': 'Generate a TRUE/FALSE/NOT GIVEN statement based on the passage.',
    'READINGYESNONOTGIVEN': 'Generate a YES/NO/NOT GIVEN statement that reflects the writer\'s opinion.',
    'READINGMULTIPLECHOICES': 'Create a multiple-choice question with 4 options (A, B, C, D).',
    'READINGSHORTANSWER': 'Create a short answer question that requires ONE WORD or A NUMBER from the passage.',
    'READINGMATCHINGHEADINGS': 'Create a heading that summarizes a paragraph.',
    'READINGMATCHINGFEATURES': 'Create a statement that can be matched with a person/theory/place mentioned in the passage.',
    'READINGMATCHINGENDINGS': 'Create a sentence opening that requires completion from the passage.',
    'READINGTEXTCOMPLETION': 'Create a summary with blanks to be filled from the passage.',
    'READINGSELECTIVETEXTCOMPLETION': 'Create a summary with multiple choice options for each blank.',
    'READINGTABLECOMPLETION': 'Create a table or form with missing information to be completed.',
}

def get_prefix_for_type(qtype: Optional[str]) -> str:
    if not qtype:
        return "Generate an IELTS reading question."
    canon_type = _canon(qtype)
    return PREFIX_MAP.get(canon_type, "Generate an IELTS reading question.")

print("✅ Prefix map defined")

✅ Prefix map defined


In [8]:
# ==== DATASET CLASS ====
@dataclass
class QGExample:
    text: str
    source_text: str  # Store source separately for evaluation
    target_text: str  # Store target separately for evaluation
    meta: Dict[str, Any]

class QGDataset(Dataset):
    """
    GPT-2 VERSION with evaluation support
    """
    
    def __init__(
        self, 
        df, 
        tokenizer, 
        max_total_len,
        default_diff="Medium",
        use_prefix=True
    ):
        self.items: List[QGExample] = []
        self.tok = tokenizer
        self.max_total_len = max_total_len
        self.default_diff = default_diff
        self.use_prefix = use_prefix
        self._build(df)
    
    def _fmt_source(self, passage, difficulty, qtype):
        """Format only the source part (for evaluation)"""
        b_value = IRT_PROXY_MAP.get(difficulty, DEFAULT_B_VALUE)
        irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"
        type_token = as_type_token(qtype)
        prefix = ""
        if self.use_prefix:
            prefix = get_prefix_for_type(qtype)
        return f"{irt_string} {prefix} {type_token}\nPassage: {passage}"
    
    def _fmt_target(self, instruction, question, answer):
        """Format only the target part (for evaluation)"""
        target_parts = []
        if instruction and instruction.strip():
            target_parts.append(f"Instruction: {instruction.strip()}")
        if question and question.strip():
            target_parts.append(f"Question: {question.strip()}")
        if answer and answer.strip():
            target_parts.append(f"Answer: {answer.strip()}")
        
        if not target_parts:
            raise ValueError("Empty target")
        return "\n".join(target_parts)
    
    def _fmt_full_text(self, source_text, target_text):
        """Combine source and target for training"""
        return f"{source_text} <SEP> {target_text} <END>"
    
    def _build(self, df):
        bad = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Building dataset"):
            passage = normalize_sep_blank(str(row["passage"]).strip())
            difficulty = str(row.get("difficulty", self.default_diff)).strip().title()
            qtype = row.get("question_type") or row.get("qtype")
            
            instruction = normalize_sep_blank(str(row.get("ref_instruction", "")).strip())
            question = normalize_sep_blank(str(row.get("ref_question", "")).strip())
            answer = normalize_sep_blank(str(row.get("ref_answer", "")).strip())
            
            # Validation
            if not instruction or not question:
                bad += 1
                continue
            
            if len(passage) < 50:
                bad += 1
                continue
            
            try:
                source_text = self._fmt_source(passage, difficulty, qtype)
                target_text = self._fmt_target(instruction, question, answer)
                full_text = self._fmt_full_text(source_text, target_text)
            except Exception as e:
                bad += 1
                continue
            
            self.items.append(QGExample(
                text=full_text,
                source_text=source_text,
                target_text=target_text,
                meta={
                    "difficulty": difficulty,
                    "question_type": qtype,
                    "passage_id": row.get("passage_id", "unknown")
                }
            ))
        
        if bad:
            print(f"⚠️ Skipped {bad} invalid rows")
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        ex = self.items[idx]
        
        # Tokenize the entire text
        tokenized = self.tok(
            ex.text,
            max_length=self.max_total_len,
            truncation=True,
            padding=False,
            return_tensors=None
        )
        
        # For GPT-2, we use the input_ids as labels as well (causal LM)
        tokenized["labels"] = tokenized["input_ids"].copy()
        
        # Store the source and target texts as separate fields (not tensors)
        # These will be removed by the custom collator
        tokenized["source_text"] = ex.source_text
        tokenized["target_text"] = ex.target_text
        tokenized["full_text"] = ex.text
        
        return tokenized

print("✅ QGDataset ready (GPT-2 Version with evaluation support)")

✅ QGDataset ready (GPT-2 Version with evaluation support)


In [9]:
# ==== CUSTOM DATA COLLATOR ====
class QGDataCollatorForLanguageModeling(DataCollatorForLanguageModeling):
    """Custom data collator that handles string fields"""
    
    def __call__(self, features):
        # Remove non-tensor fields before processing
        string_fields = ["source_text", "target_text", "full_text"]
        saved_fields = {}
        
        for field in string_fields:
            if field in features[0]:
                saved_fields[field] = [feature.pop(field) for feature in features]
        
        # Process the tensor fields with the parent collator
        batch = super().__call__(features)
        
        # Add the string fields back
        batch.update(saved_fields)
        
        return batch

In [10]:
# ==== TRAINING UTILITIES ====
def add_special_tokens(tokenizer, model, extra_tokens=None):
    base = UTILITY_TOKENS 
    if extra_tokens:
        base += list(sorted(set(t for t in extra_tokens if t)))
    
    special_tokens = {"additional_special_tokens": base}
    added = tokenizer.add_special_tokens(special_tokens)
    
    if added > 0:
        model.resize_token_embeddings(len(tokenizer))
        print(f"✅ Added {added} special tokens (total: {len(base)})")
    
    return added

In [11]:
def maybe_wrap_lora(model, config):
    if not config.get("use_lora", False):
        print("ℹ️ LoRA disabled")
        return model
    
    if not HAVE_PEFT:
        print("⚠️ PEFT not available, continuing without LoRA")
        return model
    
    print("🔧 Applying LoRA adapters for GPT-2...")
    
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=config.get("lora_r", 16),
        lora_alpha=config.get("lora_alpha", 32),
        lora_dropout=config.get("lora_dropout", 0.05),
        bias="none",
        target_modules=["c_attn", "c_proj", "c_fc"]
    )
    
    model = get_peft_model(model, lora_config)
    try:
        model.print_trainable_parameters()
    except:
        pass
    
    return model

In [12]:
@torch.no_grad()
def demo_generation(model, tokenizer, test_cases, max_total_len, device):
    """
    Test question generation with GPT-2
    """
    model.eval()
    model.to(device)
    
    print("\n" + "="*80)
    print("🎯 DEMO GENERATION - GPT-2 with IRT Proxy Control")
    print("="*80)
    
    for i, (diff, qtype, passage_snippet) in enumerate(test_cases, 1):
        
        # IRT Logic
        b_value = IRT_PROXY_MAP.get(diff, DEFAULT_B_VALUE)
        irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"

        type_token = as_type_token(qtype)
        prefix = get_prefix_for_type(qtype)
        
        # Create prompt (source part only)
        prompt = f"{irt_string} {prefix} {type_token}\nPassage: {passage_snippet} <SEP>"
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_total_len
        ).to(device)
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=CFG["max_target_len"],
            num_beams=CFG["num_beams"],
            temperature=CFG["temperature"],
            top_p=CFG["top_p"],
            length_penalty=CFG["length_penalty"],
            early_stopping=True,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
        
        # Decode only the generated part (after prompt)
        prompt_length = len(inputs["input_ids"][0])
        generated_ids = outputs[0][prompt_length:]
        result = tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        # Remove <END> token if present
        if "<END>" in result:
            result = result.split("<END>")[0].strip()
        
        print(f"\n--- Test Case {i} ---")
        print(f"Difficulty: {diff} (b={b_value})")
        print(f"Type: {qtype}")
        print(f"\nGenerated Output:")
        print(result)
        print("-" * 80)

print("✅ Training utilities ready (GPT-2 Version with metrics)")

✅ Training utilities ready (GPT-2 Version with metrics)


In [13]:
# ==== DATA LOADING AND PROCESSING ====
seed_everything(CFG["seed"])

df = load_table(CFG["data_path"])
print(f"📊 Raw data shape: {df.shape}")
print(f"📊 Columns: {list(df.columns)}")

rename_map = {
    "PassageContext": "passage",
    "QuestionType": "question_type",
    "InstructionContext": "ref_instruction",
    "Question": "ref_question",
    "Answer": "ref_answer",
}
df = df.rename(columns=rename_map)

if "passage_id" not in df.columns:
    df["passage_id"] = df.apply(stable_passage_id, axis=1)

text_cols = ["passage", "ref_instruction", "ref_question", "ref_answer", "difficulty"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: normalize_sep_blank(str(x)) if pd.notna(x) else x
        )

# Filtering
before = len(df)
df = df[df["ref_instruction"].fillna("").str.strip().ne("")]
df = df[df["ref_question"].fillna("").str.strip().ne("")]
df = df[df["passage"].fillna("").str.len() >= 50] 
after = len(df)
print(f"✅ Filtered for incomplete rows: {after}/{before} rows remaining")

# Filter question types with distribution < 50
print(f"\nFiltering by question type distribution (threshold >= 50)...")
before_dist_filter = len(df)

type_counts_map = df['question_type'].map(df['question_type'].value_counts())
df = df[type_counts_map >= 50].reset_index(drop=True)

after_dist_filter = len(df)
print(f"✅ Filtered by distribution: {after_dist_filter}/{before_dist_filter} rows remaining")

print("\n📊 Dataset Statistics:")
print(f"Difficulty distribution:\n{df['difficulty'].value_counts()}")
print(f"\nQuestion type distribution:\n{df['question_type'].value_counts()}")

cat_tokens, type_tokens = make_control_tokens_from_df(df)
extra_tokens = cat_tokens + type_tokens
print(f"\n🏷️ Control tokens: {len(extra_tokens)}")
print(f"Sample: {extra_tokens[:5]}")

# Split data
train_df, val_df, test_df = grouped_splits(df, holdout_frac=0.20, seed=CFG["seed"])
print(f"\n✅ Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

print("\n✅ Data loading complete!")

# ==== MODEL AND TOKENIZER ====
print("🔧 Loading model and tokenizer...")

tokenizer = GPT2TokenizerFast.from_pretrained(CFG["base_model"])
# Add pad token for GPT-2
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(CFG["base_model"])

add_special_tokens(tokenizer, model, extra_tokens=extra_tokens)

if CFG["grad_ckpt"]:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")

model = maybe_wrap_lora(model, CFG)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Model loaded on: {device}")


📊 Raw data shape: (2601, 10)
📊 Columns: ['Topic', 'PassageContext', 'QuestionType', 'InstructionContext', 'Question', 'Answer', 'category', 'difficulty', 'category_confidence', 'difficulty_confidence']
✅ Filtered for incomplete rows: 2596/2601 rows remaining

Filtering by question type distribution (threshold >= 50)...
✅ Filtered by distribution: 2535/2596 rows remaining

📊 Dataset Statistics:
Difficulty distribution:
difficulty
Easy      977
Medium    941
Hard      617
Name: count, dtype: int64

Question type distribution:
question_type
readingTextCompletion       986
readingShortAnswer          537
readingTrueFalseNotGiven    479
readingMultipleChoices      285
readingMatchingFeatures     156
readingYesNoNotGiven         92
Name: count, dtype: int64

🏷️ Control tokens: 6
Sample: ['<TYPE_READINGMATCHINGFEATURES>', '<TYPE_READINGMULTIPLECHOICES>', '<TYPE_READINGSHORTANSWER>', '<TYPE_READINGTEXTCOMPLETION>', '<TYPE_READINGTRUEFALSENOTGIVEN>']

✅ Split: Train=2026, Val=239, Test=270

✅ D

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ Added 9 special tokens (total: 9)
🔧 Applying LoRA adapters for GPT-2...
trainable params: 2,359,296 || all params: 126,806,016 || trainable%: 1.8606
✅ Model loaded on: cuda


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [14]:
# ==== DATASETS ====
train_ds = QGDataset(
    train_df,
    tokenizer,
    CFG["max_total_len"],
    use_prefix=True
)

val_ds = QGDataset(
    val_df,
    tokenizer,
    CFG["max_total_len"],
    use_prefix=True
)

print(f"📊 Train: {len(train_ds)}, Val: {len(val_ds)}")

Building dataset: 100%|██████████| 239/239 [00:00<00:00, 9000.49it/s]

📊 Train: 2026, Val: 239


In [15]:
# ==== DATA COLLATOR ====
collator = QGDataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

In [16]:
# ==== TRAINING ====
training_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    per_device_eval_batch_size=CFG["batch_size"],
    learning_rate=CFG["lr"],
    num_train_epochs=CFG["epochs"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=50,
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=CFG["weight_decay"],
    fp16=CFG["fp16"],
    bf16=CFG["bf16"],
    gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
    report_to=["none"],
    remove_unused_columns=False,
    dataloader_num_workers=2,
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["patience"])],
)

/tmp/ipykernel_47/328203816.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
train_output = trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.764000,3.231580
2,1.737200,3.180449
3,1.690400,3.108085
4,1.607300,3.033530
5,1.567000,2.980818
6,1.519000,2.963799
7,1.500700,2.960834
8,1.467800,2.967012
9,1.453700,2.980167
10,1.437100,2.990098


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:252: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:252: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input 

In [ ]:
print("💾 Saving model/tokenizer to:", CFG["output_dir"])
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

if len(test_df) > 0:
    print("🧪 Final test evaluation…")
    test_ds = QGDataset(test_df, tokenizer, CFG["max_source_len"], CFG["max_target_len"])
    metrics = trainer.evaluate(eval_dataset=test_ds)
    print(metrics)
else:
    print("No test split available.")

In [ ]:
if len(val_df) > 0:
    sample_df = val_df.sample(10, random_state=CFG["seed"])
    
    test_cases_list = [
        (row["difficulty"], row["question_type"], row["passage"][:1024])
        for _, row in sample_df.iterrows()
    ]
    print(f"--- Generating {len(test_cases_list)} demo samples ---")

    demo_generation(
        trainer.model, 
        tokenizer, 
        test_cases_list, 
        CFG["max_source_len"], 
        CFG["max_target_len"], 
        device
    )
else:
    print("No validation split to sample from.")

In [28]:
HUB_MODEL_ID = "alikarimiaca/controlable-question-generation-T5-base"
model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
trainer.push_to_hub(HUB_MODEL_ID)
print("\nAll done! Check your Hugging Face profile.")

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:252: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


All done! Check your Hugging Face profile.


## **Evaluation**

In [2]:
from transformers import T5ForConditionalGeneration, T5TokenizerFast

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import torch

BASE_MODEL_NAME = "t5-base"
FINETUNED_MODEL_NAME = "alikarimiaca/controlable-question-generation-T5-base"

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_NAME, use_fast=True)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)
base_model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_NAME)

model.eval()
print("✅ Fine-Tuned Model loaded successfully!")

✅ Fine-Tuned Model loaded successfully!


In [65]:
print("Tokenizer size:", len(tokenizer))
print("Embeddings:", model.get_input_embeddings().weight.shape)  # should be (len(tokenizer), hidden_size)

Tokenizer size: 32108
Embeddings: torch.Size([32108, 768])


In [20]:
import re 

DEFAULT_DIFFICULTY = "Medium"

def as_cat_token(category: Optional[str]) -> str:
    return "" 

def format_source(passage, difficulty, category=None, qtype=None):
    """Creates the model's input string (IRT Proxy Version)."""
    # IRT Logic
    b_value = IRT_PROXY_MAP.get(difficulty, DEFAULT_B_VALUE)
    irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"
    
    type_token = as_type_token(qtype)
    prefix = get_prefix_for_type(qtype)
    return f"{irt_string} {prefix} {type_token}\nPassage: {passage}"


def format_target(difficulty, instruction, question, answer, category=None, qtype=None):
    """
    This function is only for reference/comparison, not for model input.
    Using the simple format.
    """
    ins = (instruction or "").strip()
    q   = (question or "").strip()
    a   = (answer or "").strip()
    if not ins or not q:
        return None 
    
    parts = []
    if ins: parts.append(f"Instruction: {ins}")
    if q: parts.append(f"Question: {q}")
    if a: parts.append(f"Answer: {a}")
    return "\n".join(parts)

def parse_generated_output(text: str) -> Dict[str, str]:
    """Parses the model's full output string into components by splitting."""
    
    instruction = ""
    question = ""
    answer = ""
    
    # 1. Split by "Answer:"
    answer_parts = re.split(r"Answer:", text, maxsplit=1, flags=re.IGNORECASE)
    if len(answer_parts) == 2:
        main_part = answer_parts[0]
        answer = answer_parts[1].strip()
    else:
        main_part = answer_parts[0]
        answer = ""
        
    # 2. Split the main part by "Question:"
    question_parts = re.split(r"Question:", main_part, maxsplit=1, flags=re.IGNORECASE)
    if len(question_parts) == 2:
        inst_part = question_parts[0]
        question = question_parts[1].strip()
    else:
        inst_part = question_parts[0]
        question = ""
        
    # 3. Clean up the instruction part (remove "Instruction:")
    instruction = re.sub(r"^\s*Instruction:", "", inst_part, flags=re.IGNORECASE).strip()

    return {
        "instruction": instruction,
        "question": question,
        "answer": answer
    }

print("✔️ Formatting and Parsing functions defined (IRT Proxy Version for Eval).")

✔️ Formatting and Parsing functions defined (IRT Proxy Version for Eval).


In [21]:
results = []

evaluation_set = val_df.head(200) 

print(f"Generating predictions for {len(evaluation_set)} test samples...")

model.to(device)

for _, row in tqdm(evaluation_set.iterrows(), total=len(evaluation_set)):
    
    # Get reference data from the row
    difficulty = str(row.get("difficulty", DEFAULT_DIFFICULTY)).strip().title()
    category   = row.get("category", None)
    qtype      = row.get("question_type", None) if "question_type" in row.index else row.get("qtype", None)
    passage    = normalize_sep_blank(str(row["passage"]).strip())
    
    ref_instruction = normalize_sep_blank(row.get("ref_instruction", ""))
    ref_question    = normalize_sep_blank(row.get("ref_question", ""))
    ref_answer      = normalize_sep_blank(row.get("ref_answer", ""))

    # 1. Format the source input
    source_text = format_source(passage, difficulty, category, qtype)
    
    # 2. Tokenize and generate
    inputs = tokenizer(
        source_text, 
        max_length=CFG["max_source_len"], 
        truncation=True, 
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CFG["max_target_len"],
            num_beams=6,
            do_sample=True,
            temperature=0.15,
            top_p=0.95,
            num_return_sequences=3,
            return_dict_in_generate=True,
            output_scores=True,
            early_stopping=True
        )

    best_sequence_index = torch.argmax(outputs.sequences_scores) 
    best_generated_ids = outputs.sequences[best_sequence_index]
    generated_text = tokenizer.decode(best_generated_ids, skip_special_tokens=True)

    # parse output
    parsed_output = parse_generated_output(generated_text)
    
    # Store all relevant information
    results.append({
        # Reference data
        "passage_id": row["passage_id"],
        "passage": passage,
        "difficulty": difficulty,
        "category": category,
        "question_type": qtype,
        "reference_instruction": ref_instruction,
        "reference_question": ref_question,
        "reference_answer": ref_answer,
        
        # Generated data (parsed)
        "generated_instruction": parsed_output["instruction"],
        "generated_question": parsed_output["question"],
        "generated_answer": parsed_output["answer"],
        
        # Full text for debugging
        "generated_full_output": generated_text
    })

print("Generation complete.")
eval_df = pd.DataFrame(results)

Generating predictions for 200 test samples...


100%|██████████| 200/200 [07:24<00:00,  2.22s/it]

Generation complete.


In [23]:
display_df.iloc[3]['generated_full_output']

'Instruction: Reading Passage 1 has eight paragraphs, A-G . Which paragraph contains the following information? Write the correct letter, A-G , in boxes 14-18 on your answer sheet. NB You may use any letter more than once. Question: IQ tests can neither identify the processes of learning and thinking nor predict creativity. Answer: B'

In [24]:
output_filename = "evaluation_results.csv"

display_df.to_csv(output_filename, index=False)

print(f"Successfully saved results to {output_filename}")

Successfully saved results to evaluation_results.csv
